# Instagram Data Scraping
    (Need multiple accounts to avoid being blocked)


1. import necessary libraries

In [47]:
import os
import time
import pandas as pd
import requests
import torch
import torch.nn as nn
from bs4 import BeautifulSoup as Soup
from scipy.special import style
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from torch.cuda import device
from torch.distributed.pipelining import pipeline
from tqdm import tqdm
from transformers import AutoTokenizer
from transformers import pipeline

2. Login + Home Page >> need to change the account and password

In [8]:
# 抓取貼文內容--------------------
browser = webdriver.Chrome()
# 想要抓取資料的頁面網址
url = 'https://www.instagram.com/' 

# 前往頁面
browser.get(url)
# ------ 填入帳號與密碼 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.NAME, 'username')))

# ------ 網頁元素定位 ------
username_input = browser.find_element(By.NAME, "username")
password_input = browser.find_element(By.NAME, "password")
print("inputing username and password...")

# ------ 輸入帳號密碼 ------
username_input.send_keys("InstagramID")
password_input.send_keys("InstagramPassword")

# ------ 登入 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.XPATH,
'//*[@id="loginForm"]/div/div[3]/button/div')))
# ------ 網頁元素定位 ------
login_click = browser.find_element(By.XPATH, '//*[@id="loginForm"]/div/div[3]/button/div')
# ------ 點擊登入鍵 ------
login_click.click()
time.sleep(20)
print("login successfully")

# 想要抓取資料的頁面網址
url = 'https://www.instagram.com' 
subUrlList = ['wendys', 'sonicdrivein', 'mcdonalds', 'mcdonalds_switzerland','mcdonaldscanada']

for subUrl in subUrlList:
    # 前往頁面
    browser.get(url + '/' + subUrl)
    time.sleep(2)
    # 往下滑並取得新的貼文連結
    post_all = []
    postList = []

    # Get scroll height.
    last_height = browser.execute_script("return document.body.scrollHeight")
    with tqdm(total=100) as pbar:
        while True:
            # Scroll down to the bottom.
            browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            # Wait to load the page.
            time.sleep(5)
    
            soup = Soup(browser.page_source, 'lxml')
            # 尋找所有的貼文連結
        
            for rowPost in soup.select('div._ac7v'):
                for elem in rowPost.select('div a'):
                    if elem['href'] in postList:
                        continue
                    postList.append(elem['href'])
                    img_elem = elem.select('div div img')
                    temp = []
                    temp.append(elem['href'])
                    try: 
                        temp.append(img_elem[0]['alt'])
                    except:
                        temp.append("")
                    temp.append(img_elem[0]['src'])
                    post_all.append(temp)
                    
            # Calculate new scroll height and compare with last scroll height.
            new_height = browser.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
            pbar.update(1)
        
    print(subUrl+" 總共 " + str(len(post_all)) + " 篇貼文")
    
    # 存成csv
    postDataframe= pd.DataFrame(post_all)
    postDataframe.rename(columns = {0:'url', 1:'imgAlt', 2:'imgSrc'}, inplace = True)
    postDataframe.to_csv('Home_'+subUrl+'.csv', index = False)

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


inputing username and password...


KeyboardInterrupt: 

3. Login + Post Page >> need to change the account and password

In [2]:
# 抓取貼文內容--------------------
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--window-size=334,778")
browser = webdriver.Chrome(options=chrome_options)
# 想要抓取資料的頁面網址
url = 'https://www.instagram.com/' 

# 前往頁面
browser.get(url)
# ------ 填入帳號與密碼 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.NAME, 'username')))

# ------ 網頁元素定位 ------
username_input = browser.find_element(By.NAME, "username")
password_input = browser.find_element(By.NAME, "password")
print("inputing username and password...")

# ------ 輸入帳號密碼 ------
username_input.send_keys("InstagramID")
password_input.send_keys("InstagramPassword")

# ------ 登入 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.XPATH,
'//*[@id="loginForm"]/div/div[3]/button/div')))
# ------ 網頁元素定位 ------
login_click = browser.find_element(By.XPATH, '//*[@id="loginForm"]/div/div[3]/button/div')
# ------ 點擊登入鍵 ------
login_click.click()
time.sleep(15)
print("login successfully")

# 想要抓取資料的頁面網址
url = 'https://www.instagram.com' 
subUrlList = ['sonicdrivein']
# 'wendys', 'mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'mcdonalds_switzerland' ,'sonicdrivein'

for subUrl in subUrlList:
    print("parsing "+subUrl+".csv")
    # insHome = pd.read_csv('Home_'+subUrl+'.csv')
    insHome = pd.read_csv('Done_'+subUrl+'.csv')
    print("parsing "+subUrl+".csv done")
    with tqdm(total=insHome.shape[0]) as pbar:
        counter = 0
        for index, row in insHome.iterrows(): #< 922
            if '/p/' in row['url'] and counter > 406 and counter < 923:
                browser.get(url + row['url'])
                time.sleep(10)
                html = browser.page_source
                soup = Soup(html, 'lxml')
                img_elem = soup.select('img.x5yr21d')
                text_elem = soup.select('h1._ap3a')

                insHome.at[index, 'imgSrc'] = img_elem[1]['src']
                try:
                    insHome.at[index, 'imgAlt'] = text_elem[0].text
                except:
                    insHome.at[index, 'imgAlt'] = ""
                insHome.to_csv('Done_'+subUrl+'.csv', index = False)
            pbar.update(1)
            counter = counter+1
            
    insHome['imgName'] = subUrl+'_'+insHome.index.astype(str)
    insHome.to_csv('Done_'+subUrl+'.csv', index = False)

inputing username and password...
login successfully
parsing mcdonalds_switzerland.csv
parsing mcdonalds_switzerland.csv done


100%|██████████| 2087/2087 [1:20:06<00:00,  2.30s/it] 


4. Download Image (from home page)

In [ ]:
subUrlList = ['wendys','mcdonalds', 'mcdonalds_switzerland','mcdonaldscanada','sonicdrivein']

for subUrl in subUrlList:
    print("parsing "+subUrl+".csv")
    insDone = pd.read_csv('Done_'+subUrl+'.csv')
    print("parsing "+subUrl+".csv done")
    with tqdm(total=insDone.shape[0]) as pbar:
        for index, row in insDone.iterrows():
            #create img folder
            if not os.path.exists(subUrl+'_img'):
                os.makedirs(subUrl+'_img')
            with open(subUrl+'_img/'+row['imgName']+'.jpg', 'wb') as f:
                try :
                    f.write(requests.get(row['imgSrc']).content)
                except:
                    print('Failed to download' + row['imgName']+'.jpg')
            pbar.update(1)

parsing wendys.csv
parsing wendys.csv done


100%|██████████| 371/371 [00:06<00:00, 54.63it/s]


parsing mcdonalds.csv
parsing mcdonalds.csv done


100%|██████████| 307/307 [00:10<00:00, 28.52it/s]


parsing mcdonalds_switzerland.csv
parsing mcdonalds_switzerland.csv done


...

100%|██████████| 850/850 [00:15<00:00, 53.73it/s]


parsing sonicdrivein.csv
parsing sonicdrivein.csv done


 23%|██▎       | 514/2240 [01:42<06:35,  4.37it/s]

Failed to downloadsonicdrivein_513.jpg


 78%|███████▊  | 1757/2240 [06:08<03:22,  2.38it/s]

5. Download Image (from IMAGE post page)  >> need to change the account and password

In [33]:
# 抓取貼文內容--------------------
browser = webdriver.Chrome()
# 想要抓取資料的頁面網址
url = 'https://www.instagram.com/' 

# 前往頁面
browser.get(url)
# ------ 填入帳號與密碼 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.NAME, 'username')))

# ------ 網頁元素定位 ------
username_input = browser.find_element(By.NAME, "username")
password_input = browser.find_element(By.NAME, "password")
print("inputing username and password...")

# ------ 輸入帳號密碼 ------
username_input.send_keys("InstagramID")
password_input.send_keys("InstagramPassword")

# ------ 登入 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.XPATH,
'//*[@id="loginForm"]/div/div[3]/button/div')))
# ------ 網頁元素定位 ------
login_click = browser.find_element(By.XPATH, '//*[@id="loginForm"]/div/div[3]/button/div')
# ------ 點擊登入鍵 ------
login_click.click()
time.sleep(20)
print("login successfully")

# 想要抓取資料的頁面網址
url = 'https://www.instagram.com' 
subUrlList = ['wendys', 'sonicdrivein', 'mcdonalds', 'mcdonalds_switzerland','mcdonaldscanada']
post_count=[371, 2240, 307, 2087, 850]
counter = -1
for subUrl in subUrlList:
    download_counter = 0
    counter = counter+1
    print("parsing "+subUrl+".csv")
    insDone = pd.read_csv('Done_'+subUrl+'.csv')
    print("parsing "+subUrl+".csv done")
    
    # 前往頁面
    browser.get(url + '/' + subUrl)
    time.sleep(2)
    # 往下滑並取得新的貼文連結
    post_all = []
    postList = []

    # Get scroll height.
    last_height = browser.execute_script("return document.body.scrollHeight")
    with tqdm(total=post_count[counter]) as pbar:
        while True:
            # Scroll down to the bottom.
            browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            # Wait to load the page.
            time.sleep(5)
    
            soup = Soup(browser.page_source, 'lxml')
            # 尋找所有的貼文連結
        
            for rowPost in soup.select('div._ac7v'):
                for elem in rowPost.select('div a'):
                    if elem['href'] in postList:
                        continue
                        
                    imgName = insDone[insDone['url'] == elem['href']]['imgName']
                    if imgName.shape[0] > 0:
                        img_elem = elem.select('div div img')
                        imgUrl = img_elem[0]['src']
                        if not os.path.exists(subUrl+'_img'):
                            os.makedirs(subUrl+'_img')
                        with open(subUrl+'_img/'+imgName.values[0]+'.jpg', 'wb') as f:
                            try :
                                f.write(requests.get(imgUrl).content)
                                download_counter = download_counter+1
                            except:
                                print('Failed to download' + imgName+'.jpg')
                        postList.append(elem['href'])
                    pbar.update(1)
            # Calculate new scroll height and compare with last scroll height.
            new_height = browser.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
            
    print(subUrl+" 總共 " + str(download_counter) + " 篇圖片已下載")


inputing username and password...
login successfully
parsing wendys.csv
parsing wendys.csv done


377it [02:55,  2.15it/s]                         


wendys 總共 371 篇圖片已下載
parsing sonicdrivein.csv
parsing sonicdrivein.csv done


2301it [17:40,  2.17it/s]                          


sonicdrivein 總共 2219 篇圖片已下載
parsing mcdonalds.csv
parsing mcdonalds.csv done


...

mcdonalds 總共 307 篇圖片已下載
parsing mcdonalds_switzerland.csv
parsing mcdonalds_switzerland.csv done


2202it [16:28,  2.23it/s]                          


mcdonalds_switzerland 總共 2050 篇圖片已下載
parsing mcdonaldscanada.csv
parsing mcdonaldscanada.csv done


877it [06:41,  2.18it/s]                         

mcdonaldscanada 總共 844 篇圖片已下載


6. Download Image (from REEL post page)  >> need to change the account and password

In [4]:
# 抓取貼文內容--------------------
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--window-size=334,778")
browser = webdriver.Chrome(options=chrome_options)
# 想要抓取資料的頁面網址
url = 'https://www.instagram.com/' 

# 前往頁面
browser.get(url)
# ------ 填入帳號與密碼 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.NAME, 'username')))

# ------ 網頁元素定位 ------
username_input = browser.find_element(By.NAME, "username")
password_input = browser.find_element(By.NAME, "password")
print("inputing username and password...")

# ------ 輸入帳號密碼 ------
username_input.send_keys("InstagramID")
password_input.send_keys("InstagramPassword")

# ------ 登入 ------
WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.XPATH,
'//*[@id="loginForm"]/div/div[3]/button/div')))
# ------ 網頁元素定位 ------
login_click = browser.find_element(By.XPATH, '//*[@id="loginForm"]/div/div[3]/button/div')
# ------ 點擊登入鍵 ------
login_click.click()
time.sleep(15)
print("login successfully")

# 想要抓取資料的頁面網址
url = 'https://www.instagram.com' 


for subUrl in subUrlList:
    print("parsing "+subUrl+".csv")
    # insHome = pd.read_csv('Home_'+subUrl+'.csv')
    insHome = pd.read_csv('Done_'+subUrl+'.csv')
    print("parsing "+subUrl+".csv done")
    with tqdm(total=insHome.shape[0]) as pbar:
        counter = 0
        for index, row in insHome.iterrows(): #< 922
            if '/p/' in row['url'] and counter > 406 and counter < 923:
                browser.get(url + row['url'])
                time.sleep(10)
                html = browser.page_source
                soup = Soup(html, 'lxml')
                img_elem = soup.select('img.x5yr21d')
                text_elem = soup.select('h1._ap3a')

                insHome.at[index, 'imgSrc'] = img_elem[1]['src']
                
                    if '/reel/' in elem['href'] and imgName.shape[0] > 0:
                        img_elem = elem.select('div div img')
                        imgUrl = img_elem[0]['src']
                        if not os.path.exists(subUrl+'_img'):
                            os.makedirs(subUrl+'_img')
                        with open(subUrl+'_img/'+imgName.values[0]+'.jpg', 'wb') as f:
                            try :
                                f.write(requests.get(imgUrl).content)
                                download_counter = download_counter+1
                            except:
                                print('Failed to download' + imgName+'.jpg')
                        postList.append(elem['href'])
                    pbar.update(1)
            pbar.update(1)
            counter = counter+1
            
    insHome['imgName'] = subUrl+'_'+insHome.index.astype(str)
    insHome.to_csv('Done_'+subUrl+'.csv', index = False)

parsing sonicdrivein.csv
parsing sonicdrivein.csv done


100%|██████████| 2240/2240 [14:36<00:00,  2.56it/s] 


# Remove Nan from Oxford

In [4]:
import pandas as pd
dirPath = '../Oxford_HIC/Original_File/oxford_hic_data.csv'
data = pd.read_csv(dirPath)
print(data.shape)
data = data.dropna()
print(data.shape)
data = data.reset_index(drop=True)
data.to_csv('../Oxford_HIC/oxford_hic_data.csv', index = False)

C:\Users\USER\AppData\Local\Temp\ipykernel_12588\176095774.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(dirPath)


(3657847, 3)
(3653781, 3)


In [6]:
data

,image_id,caption,funny_score
0,bokete_0,My driver's license photo,0.0
1,bokete_1,Refugee relief.,0.0
2,bokete_2,Now! I think I stepped on a cat! What? Really?...,0.0
3,bokete_3,You wouldn't know I was reading a comic book.,0.0
4,bokete_4,"Oh no! I forgot my ・・・・ clothes!""",0.0
...,...,...,...
3653776,british-high-school-honeybee,on time,0
3653777,british-high-school-honeybee,quiz tomorrow bee ready!!!,0
3653778,british-high-school-honeybee,i'll die no matter what my fucking defenses bitch,0
3653779,british-high-school-honeybee,you dont scare me biker dude,0


In [2]:
import pandas as pd
from extractor import addImagePath
import tqdm
import os
dirPath = '../Oxford_HIC/Original_File/oxford_hic_data.csv'
imgPath = '../Oxford_HIC/oxford_img/'
data = pd.read_csv(dirPath)
print(data.shape)
new_data = addImagePath(data, imgPath)
# if image is not exist, remove the row
with tqdm.tqdm(total=data.shape[0]) as pbar:
    for i in range(data.shape[0]):
        if not os.path.exists(data['image_id'][i]):
            data = data.drop(i)
        pbar.update(1)
print(data.shape)
data = data.reset_index(drop=True)
# data.to_csv('../Oxford_HIC/oxford_hic_data.csv', index=False)

C:\Users\USER\AppData\Local\Temp\ipykernel_15004\3786780886.py:7: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(dirPath)


(3657847, 3)


 89%|████████▊ | 3243763/3657847 [01:48<00:13, 30014.97it/s]


KeyboardInterrupt: 

In [2]:
data.to_csv('../Oxford_HIC/Filter_oxford_hic_data.csv', index=False)

In [4]:
print(3653781 - 3400257)

253524


In [41]:
import pandas as pd
dirPath = '../Oxford_HIC/Filtered_oxford_hic_data.csv'
imgPath = '../Oxford_HIC/oxford_img/'

data = pd.read_csv(dirPath)
# data
# data['image_id'] = data['image_id'].apply(lambda x: x.split('/')[-1].split('.')[0])
data["funny_score"] = data["funny_score"].apply(lambda x: int(float(x.replace(',', ''))) if type(x) == str else int(float(x)))
data["funny_score"] = data["funny_score"]/data["funny_score"].max()

C:\Users\USER\AppData\Local\Temp\ipykernel_15004\3302334520.py:7: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(dirPath)


In [44]:
data.to_csv('../Oxford_HIC/Filtered_oxford_hic_data.csv', index=False)

In [3]:
import pandas as pd
dirPath = '../Oxford_HIC/Filtered_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
# remove caption is imgflip_118
print(data.shape)
# data = data[data['image_id'] != 'imgflip_118']  # (3400257, 3) > (3398346, 3)
# data = data[data['image_id'] != 'angryschoolboy']  # (3398346, 3) > (3398160, 3)
data = data[data['image_id'] != 'bokete_966']  
data = data[data['image_id'] != 'bokete_2495']  
data = data[data['image_id'] != 'bokete_3049']  
data = data[data['image_id'] != 'bokete_6997']  
data = data[data['image_id'] != 'bokete_7487']  
data = data[data['image_id'] != 'bokete_8176']
data = data[data['image_id'] != 'bokete_12501']  
data = data[data['image_id'] != 'bokete_25616']  
data = data[data['image_id'] != 'bokete_67874']  
data = data[data['image_id'] != 'bokete_68584']  
data = data[data['image_id'] != 'bokete_69659']  
data = data[data['image_id'] != 'bokete_70929']
data = data[data['image_id'] != 'bokete_12501']  # (3398160, 3) > (3398081, 3)
print(data.shape)

(3398160, 3)
(3398081, 3)


In [4]:
data.to_csv('../Oxford_HIC/Filtered_oxford_hic_data.csv', index=False)

In [6]:
import pandas as pd
dirPath = '../Oxford_HIC/Filtered_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
# remove caption is imgflip_118
print(data.shape)

(3398346, 3)
